# Concise build orchestrator -- median baseline vs LightGBM

Runs, in dependency order, the stages and pipeline arms this `concise/`
folder actually carries:

1. `stage1_audit.py` -- data audit + AOBT_3 near-target-proxy leakage check
2. `stage2_split.py` -- departure-only, time-based (`forward`) train/val/test split
3. `pipeline_naive_offblock/` -- median-hierarchy / AOBT-proxy baseline
4. `pipeline_lightgbm/` -- basic LightGBM model
5. `stage6_clip_sweep.py` -- sweeps the prediction-clip cap for `lightgbm` on val
6. A side-by-side comparison on the **val** split, with the val-selected winner's **test** RMSE

Each step's non-zero exit halts the run, the same contract as
`solution/run_all.py` -- a stage does not run on a gate that didn't pass.

Protocol used: `config.DEFAULT_SPLIT` / `config.DEFAULT_FEATURESET` /
`config.DEFAULT_OUTLIER_POLICY` (currently `forward` / `with_aobt` / `keep_all`),
seed 42 -- the same defaults the full `solution/` build treats as its primary
protocol, not a choice made here.

Every stage still writes its artifact JSON under `artifacts/` (so results are
reproducible and diffable outside the notebook), but each cell's own printed
report is also captured into that cell's output below -- so reading this
notebook top to bottom is enough; nobody needs to go open the artifact files
to see what a stage found.


In [1]:
from __future__ import annotations

import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "config.py").exists(), f"run this notebook from concise/, not {ROOT}"
sys.path.insert(0, str(ROOT))
import config

SPLIT = config.DEFAULT_SPLIT
FEATURESET = config.DEFAULT_FEATURESET
POLICY = config.DEFAULT_OUTLIER_POLICY
SEED = 42
BASE_ARGS = ["--split", SPLIT, "--featureset", FEATURESET,
             "--outlier-policy", POLICY, "--seed", str(SEED)]
print(f"protocol: split={SPLIT} featureset={FEATURESET} outlier_policy={POLICY} seed={SEED}")


def run(cmd: "list[str]") -> None:
    """Run a stage script and stream its stdout/stderr into THIS cell's output.

    subprocess.call() with no stdout= argument inherits the OS-level file
    descriptor, which bypasses Jupyter's stdout capture -- so every stage's own
    report (audit gates, split summary, RMSE/MAE breakdown) was being written to
    the terminal the kernel was launched from, not saved into the notebook.
    Piping stdout and re-printing line by line puts it through Python's
    sys.stdout instead, which Jupyter does capture, so the result lands in the
    cell (and gets saved in the .ipynb) instead of only existing as a file under
    artifacts/.
    """
    print("\n" + "=" * 78)
    print("$ " + " ".join(cmd))
    print("=" * 78, flush=True)
    t0 = time.time()
    proc = subprocess.Popen([sys.executable] + cmd, cwd=str(ROOT),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"[{time.time() - t0:.1f}s, rc={rc}]")
    if rc != 0:
        raise SystemExit(
            f"HALT: `{' '.join(cmd)}` failed with rc={rc}. This stage's gate did "
            "not pass -- the next stage must not run on its output.")


protocol: split=forward featureset=with_aobt outlier_policy=keep_all seed=42


## Stage 1 -- data audit

Includes the AOBT_3 near-target-proxy check (target/timestamp leakage).

In [2]:
run(["stage1_audit.py", "--force"])


$ stage1_audit.py --force


== files ==


  12 training files, 4167797 rows total


  train DEP 2085047 | ranking DEP 215876 | submitting 215876


== G2: ranking-availability contract ==


...

== numeric drift: hour-of-day and taxi-in (ARR) ==


  ARR taxi-in mean: train 536.7s -> ranking 535.0s


== verdict ==


  all gates PASS. report: C:\Users\Admin\Desktop\eddie\fearless-barn\concise\artifacts\audit\audit_report.json


[36.0s, rc=0]


## Stage 2 -- departure-only, time-based split

In [3]:
run(["stage2_split.py", "--split", SPLIT, "--force"])


$ stage2_split.py --split forward --force


== split 'forward' ==


  train n= 1387414 months=['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08'] airports=['EDDF', 'EDDM', 'EGLL', 'EHAM', 'LEBL', 'LEMD', 'LFPG', 'LIRF', 'LSZH', 'LTFM']


  val   n=  369624 months=['2025-09', '2025-10'] airports=['EDDF', 'EDDM', 'EGLL', 'EHAM', 'LEBL', 'LEMD', 'LFPG', 'LIRF', 'LSZH', 'LTFM']


  test  n=  328009 months=['2025-11', '2025-12'] airports=['EDDF', 'EDDM', 'EGLL', 'EHAM', 'LEBL', 'LEMD', 'LFPG', 'LIRF', 'LSZH', 'LTFM']


== split gates ==


  [PASS] pairwise ID disjointness (2085047 ids total)


  [PASS] month exclusivity, non-empty sets, airport filter applied


  all split gates PASS -> C:\Users\Admin\Desktop\eddie\fearless-barn\concise\artifacts\splits\forward


[26.4s, rc=0]


## Arm 1 -- median / AOBT-proxy baseline (`pipeline_naive_offblock`)

In [4]:
run(["pipeline_naive_offblock/train.py"] + BASE_ARGS + ["--force"])
for part in ("val", "test"):
    run(["-m", "pipeline_naive_offblock.evaluate"] + BASE_ARGS + ["--part", part, "--force"])



$ pipeline_naive_offblock/train.py --split forward --featureset with_aobt --outlier-policy keep_all --seed 42 --force


[naive_offblock] fitted on n=1387105 -> C:\Users\Admin\Desktop\eddie\fearless-barn\concise\artifacts\runs\naive_offblock__forward-with_aobt-keep_all__seed42


   table stand_runway   levels=6348


   table adep_stand     levels=2157


   table adep_runway    levels=59


...

    LSZH   n=  20663 rmse=  254.75 mean_true=   757.2


    EDDF   n=  35446 rmse=  251.22 mean_true=   892.3


    LEMD   n=  35404 rmse=  246.54 mean_true=   998.6


  prediction source mix: {'aobt_proxy': 0.99084, 'stand_runway': 0.00852, 'adep_stand': 0.00055, 'adep_runway': 9e-05, 'adep': 1e-05}


[7.2s, rc=0]


## Arm 2 -- basic LightGBM (`pipeline_lightgbm`)

In [5]:
run(["pipeline_lightgbm/train.py"] + BASE_ARGS + ["--force"])
for part in ("val", "test"):
    run(["-m", "pipeline_lightgbm.evaluate"] + BASE_ARGS + ["--part", part, "--force"])



$ pipeline_lightgbm/train.py --split forward --featureset with_aobt --outlier-policy keep_all --seed 42 --force


[lightgbm] train n=1387105 val n=369624 feats=57 cats=15 device=cpu


Training until validation scores don't improve for 200 rounds


[100]	train's rmse: 329.901	val's rmse: 307.77


[200]	train's rmse: 300.234	val's rmse: 303.111


...

    LEBL   n=  27167 rmse=  189.67 mean_true=   921.0


    LEMD   n=  35404 rmse=  177.83 mean_true=   998.6


    EHAM   n=  38690 rmse=  173.61 mean_true=   777.2


    LSZH   n=  20663 rmse=  171.29 mean_true=   757.2


[13.4s, rc=0]


## Stage 6 -- prediction-cap sweep (`lightgbm` only)

Sweeps `config.PREDICTION_CLIP_MAX_SEC` -- the ceiling predictions are clipped
to -- against val RMSE, per design-gate note in `config.py`: the cap is a
hyperparameter, not a fact about the data. Only `lightgbm`/`ridge` expose
unclipped predictions for this stage, and `ridge` isn't one of the two arms
this folder carries, so this runs against `lightgbm` (the default).

In [6]:
run(["stage6_clip_sweep.py"] + BASE_ARGS + ["--arm", "lightgbm", "--force"])


$ stage6_clip_sweep.py --split forward --featureset with_aobt --outlier-policy keep_all --seed 42 --arm lightgbm --force


== cap sweep: lightgbm / forward / with_aobt / keep_all / seed 42 (val, n=369624) ==


        cap   val RMSE     in-band     y>cap  pred>cap  hard floor


       7200     337.82      238.23        93        52      233.89


       9000     331.73      239.77        56        37      221.82


...

      43200     302.45      252.90         4         0       85.08 <-- best


      86400     302.45      252.90         1         0        1.88 (current)


       none     302.45      252.90         0         0        0.00


  best cap 43200 beats the current 86400 s cap by 0.00 s of val RMSE, decided by 1 val rows above the current cap.


[10.7s, rc=0]


## Compare

Not `compare.py` from `solution/` -- that script iterates `config.ARMS` (all 18
arms of the full build) and would fail looking for run directories this folder
doesn't have. This is the same val-selects-winner, test-is-reported-only-for-
the-winner logic, scoped to just these two arms.


In [7]:
import json

from common.bootstrap import run_dir

ARMS = ["naive_offblock", "lightgbm"]
proto = config.protocol_key(SPLIT, FEATURESET, POLICY)
rows = []
for arm in ARMS:
    d = run_dir(config.ARTIFACTS / "runs", arm, protocol=proto, seed=SEED)
    val = json.loads((d / "eval_val.json").read_text())
    test = json.loads((d / "eval_test.json").read_text())
    rows.append({"arm": arm, "val_rmse": val["rmse"], "test_rmse": test["rmse"]})

rows.sort(key=lambda r: r["val_rmse"])
winner = rows[0]

print(f"{'arm':<16}{'val_rmse':>12}{'test_rmse':>12}")
for r in rows:
    mark = "  <- selected on val" if r["arm"] == winner["arm"] else ""
    print(f"{r['arm']:<16}{r['val_rmse']:>12.2f}{r['test_rmse']:>12.2f}{mark}")

print(f"\nwinner (selected on val, NOT test): {winner['arm']}")
print(f"winner's test RMSE: {winner['test_rmse']:.2f}")


arm                 val_rmse   test_rmse
lightgbm              302.45      310.63  <- selected on val
naive_offblock        481.97      484.76

winner (selected on val, NOT test): lightgbm
winner's test RMSE: 310.63


## Notes

- Selection is on **val**, and the number reported for the winner is its
  **test** RMSE -- the winner is never chosen by looking at test (E3 leak gate).
- Stage 6's cap sweep is also selected on val only; it never touches test.
- To repeat with a different protocol, change `SPLIT` / `FEATURESET` / `POLICY`
  / `SEED` in the first code cell and re-run from Stage 2 -- Stage 1's audit
  report does not depend on the protocol.
